# Etapa 2: los modelos de la etapa 1 con el corpus bilingüe

Entrena con HumAID en inglés y HumAID traducido al español (`data/processed/bilingue/`, ver `scripts/prepare_humaid_es.py` y `scripts/prepare_bilingual.py`). **La arquitectura y los hiperparámetros son los que eligió la etapa 1 con val**; solo cambian los datos, así que la diferencia entre etapas mide el efecto de agregar español traducido.

Contenido:

1. BiLSTM con la configuración final de la etapa 1.
2. El Transformer ganador del screening de la etapa 1, misma receta LP-FT.
3. Comparación con la etapa 1.

BERTweet no se reentrena: solo maneja inglés. Todo se entrena con la semilla 42, como en la etapa 1, y los modelos finales se guardan en `models/etapa2/`.

## 1. Repositorio, dependencias y datos

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JuanTrujilloM/final-project-NN-DL.git"
BRANCH = "main"

# On Kaggle the repo is cloned; locally the notebook runs from the repo root.
if Path("/kaggle").exists():
    if not Path("final-project-NN-DL").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("final-project-NN-DL")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not Path("data/processed/humaid_en/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print("Directorio de trabajo:", Path.cwd())

In [ ]:
import shutil

import pandas as pd

TRANSLATION_DIR = Path("data/raw/humaid_events_translated")
TRANSLATION_FILES = ["train.csv", "val.csv", "test.csv", "test_intra.csv"]


# Finds the folder with the four translated CSVs among the Kaggle inputs.
def find_translation(root=Path("/kaggle/input")):
    for path in sorted(root.rglob("test_intra.csv")):
        folder = path.parent
        if all((folder / name).exists() for name in TRANSLATION_FILES):
            if "text_es" in pd.read_csv(folder / "train.csv", nrows=1).columns:
                return folder
    return None


if not (TRANSLATION_DIR / "train.csv").exists() and Path("/kaggle/input").exists():
    source = find_translation()
    if source is not None:
        TRANSLATION_DIR.mkdir(parents=True, exist_ok=True)
        for name in TRANSLATION_FILES:
            shutil.copy(source / name, TRANSLATION_DIR / name)
        print("Traducción copiada desde", source)

if not (TRANSLATION_DIR / "train.csv").exists():
    raise SystemExit("No encuentro la traducción: agrega humaid_es.zip como Dataset de Kaggle o cópiala a "
                     f"{TRANSLATION_DIR}")
if not Path("data/processed/humaid_es/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_humaid_es.py"], check=True)
if not Path("data/processed/bilingue/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_bilingual.py"], check=True)

## 2. Librerías y configuración

In [ ]:
import gc
import io
import json
import platform
import random
import re
import shutil
import time
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    os.environ["PL_GLOBAL_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

In [ ]:
DATA_DIR = Path("data/processed")
BI_DIR = DATA_DIR / "bilingue"
RESULTS_DIR = Path("results/etapa2")
STAGE1_DIR = Path("results/etapa1")
MODELS_DIR = Path("models") / "checkpoints" / "etapa2"
EXPORT_DIR = Path("models") / "etapa2"
GLOVE_DIR = Path("data/raw/glove")
TEXT_COL = "text_clean"

# None to train with all data; a small number (e.g. 500) for a quick test.
LIMIT = None
# Skips runs that already have a resumen.json (useful if the session drops).
SKIP_DONE = True
# What to run; each block can go in a different session.
RUN = {"bilstm": True, "transformer": True}

LABEL_NAMES = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "not_humanitarian",
    "other_relevant_information",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
]
NUM_CLASSES = len(LABEL_NAMES)
LABEL2ID = {label: i for i, label in enumerate(LABEL_NAMES)}
for path in [RESULTS_DIR, MODELS_DIR, EXPORT_DIR, GLOVE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## 3. Datos

- **Train**: `humaid_en` y `humaid_trad` (su traducción).
- **Val** (early stopping y selección del mejor modelo): inglés y traducido juntos, para que el modelo no se ajuste a un solo idioma.
- **Evaluación**: `val`, `test` y `test_intra` son los mismos desastres de la etapa 1, ahora con inglés y traducido juntos en un solo conjunto, igual que train. `chile_2014` (español nativo) no cambia.

In [ ]:
def read_csv(path):
    df = pd.read_csv(path, dtype={"tweet_id": str})
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    return df


def limit_rows(df, by=None):
    if LIMIT is None:
        return df.reset_index(drop=True)
    if by is None:
        return df.sample(n=min(LIMIT, len(df)), random_state=SEED).reset_index(drop=True)
    return df.sample(frac=1, random_state=SEED).groupby(by).head(LIMIT).reset_index(drop=True)


train_df = limit_rows(read_csv(BI_DIR / "train.csv"), by="fuente")
# val, test and test_intra keep English and Spanish together, like train.
eval_dfs = {role: limit_rows(read_csv(BI_DIR / f"{role}.csv"), by="lang") for role in ["val", "test", "test_intra"]}
eval_dfs["chile_2014"] = limit_rows(read_csv(DATA_DIR / "chile2014_es" / "test.csv"))
val_df = eval_dfs["val"]

# The corpus script already checks this; it is repeated here so the notebook never trains with a leak.
eval_ids = set().union(*(set(df["tweet_id"]) for df in eval_dfs.values()))
assert not set(train_df["tweet_id"]) & eval_ids, "Hay tweets de evaluación en train"

print(train_df.groupby("fuente").size().to_string())
for name, df in eval_dfs.items():
    print(f"  {name}: {len(df):,} ({df['lang'].value_counts().to_dict() if 'lang' in df else 'es'})")

### 3.1 Decisiones de la etapa 1

Arquitectura e hiperparámetros que eligió la etapa 1 con val:

- `BILSTM_HP`: configuración final del BiLSTM después de las ablaciones (`results/etapa1/seleccion/config_desde_cero.json`).
- `FT_KEY` y `FT_HP`: Transformer ganador del screening y su configuración (`results/etapa1/seleccion/config_transformer.json`).

Se mantienen fijos entre etapas, así que la diferencia de resultados mide solo el efecto de los datos.

In [ ]:
BILSTM_HP = {
    "hidden_size": 128,
    "num_layers": 2,
    "rnn_dropout": 0.3,
    "pooling": "attention_max",
    "embeddings": "glove",
    "freeze_embeddings": True,
    "max_len": 80,
    "emb_dropout": 0.3,
    "dropout": 0.5,
    "head_hidden": 256,
    "batch_size": 64,
    "epochs": 50,
    "lr": 0.001,
    "weight_decay": 0.01,
    "clip_grad": 1.0,
    "patience": 5,
    "lr_patience": 2,
    "lr_factor": 0.5,
    "class_weights": "sqrt",
}

FT_KEY = "mdeberta_base"
FT_HP = {
    "model_name": "microsoft/mdeberta-v3-base",
    "max_len": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "lp_lr": 0.001,
    "lp_epochs": 20,
    "lp_patience": 3,
    "lr": 2e-05,
    "llrd": 0.95,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "epochs": 10,
    "patience": 3,
    "clip_grad": 1.0,
    "class_weights": "balanced",
    "amp": True,
}

## 4. Métricas y reporte

In [ ]:
# F1 over the classes present in the true labels.
def compute_metrics(labels, preds):
    present = sorted(set(labels))
    return {
        "f1_macro": f1_score(labels, preds, labels=present, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, labels=present, average="weighted", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


# Saves metrics, per-class report and confusion matrix for one set.
def full_report(labels, preds, name, model_name, out_dir):
    present = sorted(set(labels))
    names = [LABEL_NAMES[i] for i in present]
    metrics = compute_metrics(labels, preds)
    metrics["n"] = len(labels)
    metrics["por_clase"] = classification_report(
        labels, preds, labels=present, target_names=names, output_dict=True, zero_division=0
    )
    (out_dir / f"metricas_{name}.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

    print(f"\n{'=' * 60}\n  {model_name} | {name} — F1-macro: {metrics['f1_macro']:.4f} | "
          f"F1 ponderado: {metrics['f1_weighted']:.4f} | accuracy: {metrics['accuracy']:.4f}\n{'=' * 60}")
    print(classification_report(labels, preds, labels=present, target_names=names, digits=4, zero_division=0))

    cm = confusion_matrix(labels, preds, labels=present)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    for ax, data, fmt, title in zip(
        axes, [cm, cm_norm], ["d", ".2f"], ["Conteos absolutos", "Proporción por clase (recall)"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_title(f"{model_name} | {name}\n{title}")
    plt.tight_layout()
    fig.savefig(out_dir / f"confusion_{name}.png", dpi=120)
    plt.show()
    plt.close(fig)
    return metrics

## 5. Callbacks

In [ ]:
# Saves the best model according to a monitored metric.
class ModelCheckpoint:
    def __init__(self, filepath, monitor="val_f1_macro", mode="max", verbose=True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float("-inf") if mode == "max" else float("inf")
        self.mode = mode

    def step(self, metrics, model):
        value = metrics[self.monitor]
        improved = value > self.best if self.mode == "max" else value < self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Mejor modelo guardado ({self.monitor}: {value:.4f})")
        return improved


# Stops training when the metric stops improving.
class EarlyStopping:
    def __init__(self, monitor="val_f1_macro", patience=5, min_delta=1e-4, mode="max"):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else float("-inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  ⏹ EarlyStopping activado (sin mejora en {self.patience} épocas)")
        return improved


# Lowers the learning rate when the metric plateaus.
class ReduceLROnPlateau:
    def __init__(self, optimizer, monitor="val_f1_macro", patience=3, factor=0.5, mode="max"):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode=mode, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

## 6. Utilidades de entrenamiento

In [ ]:
# Class weights from train: balanced, sqrt (softer) or none.
def class_weights(label_ids, mode="balanced"):
    if mode == "none":
        return None
    counts = np.bincount(np.asarray(label_ids), minlength=NUM_CLASSES).astype(float)
    weights = len(label_ids) / (NUM_CLASSES * np.maximum(counts, 1))
    if mode == "sqrt":
        weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)


# One epoch; trains if an optimizer is given, otherwise only evaluates.
def run_epoch(model, loader, criterion, forward_fn, optimizer=None, scheduler=None, scaler=None, clip_grad=1.0):
    training = optimizer is not None
    model.train() if training else model.eval()
    use_amp = scaler is not None
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits, labels = forward_fn(model, batch)
                loss = criterion(logits.float(), labels)

            if training:
                optimizer.zero_grad()
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / n, metrics["accuracy"], metrics["f1_macro"]


@torch.no_grad()
def evaluate_clf(model, loader, forward_fn):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits, labels = forward_fn(model, batch)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels


def new_history():
    return {"epoch": [], "phase": [], "lr": [], "train_loss": [], "val_loss": [],
            "train_acc": [], "val_acc": [], "train_f1_macro": [], "val_f1_macro": []}


def log_epoch(history, epoch, phase, lr, train, val):
    for key, value in [("epoch", epoch), ("phase", phase), ("lr", lr),
                       ("train_loss", train[0]), ("train_acc", train[1]), ("train_f1_macro", train[2]),
                       ("val_loss", val[0]), ("val_acc", val[1]), ("val_f1_macro", val[2])]:
        history[key].append(value)
    print(f"Época {epoch:02d} [{phase}] — "
          f"train loss: {train[0]:.4f}, F1-macro: {train[2]:.4f} | "
          f"val loss: {val[0]:.4f}, F1-macro: {val[2]:.4f}, acc: {val[1]:.4f} | lr: {lr:.2e}")
    return {"val_loss": val[0], "val_acc": val[1], "val_f1_macro": val[2]}


def plot_history(history, model_name, out_dir):
    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val", linestyle="--")
    axes[1].plot(epochs, history["train_f1_macro"], label="train")
    axes[1].plot(epochs, history["val_f1_macro"], label="val", linestyle="--")
    axes[0].set_title(f"{model_name} — Loss")
    axes[1].set_title(f"{model_name} — F1-macro")
    for ax in axes:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / "curvas.png", dpi=120)
    plt.show()
    plt.close(fig)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "entrenables": trainable}


def hardware_info(start_time):
    info = {
        "tiempo_entrenamiento_min": round((time.time() - start_time) / 60, 2),
        "plataforma": platform.platform(),
        "torch": torch.__version__,
        "cpu_nucleos": os.cpu_count(),
        "gpu": None,
    }
    if DEVICE.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        info["gpu"] = props.name
        info["gpu_memoria_gb"] = round(props.total_memory / 1e9, 1)
        info["gpu_memoria_max_usada_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return info


# Evaluates the best model on every eval set; predictions are saved without tweet text.
def final_evaluation(model, make_loader, forward_fn, model_name, out_dir):
    results = {}
    for name, df in eval_dfs.items():
        preds, labels = evaluate_clf(model, make_loader(df), forward_fn)
        metrics = full_report(labels, preds, name, model_name, out_dir)
        results[name] = {key: metrics[key] for key in ("f1_macro", "f1_weighted", "accuracy", "n")}
        pd.DataFrame({
            "tweet_id": df["tweet_id"].astype(str),
            "event": df["event"],
            "lang": df["lang"],
            "label": [LABEL_NAMES[i] for i in labels],
            "pred": [LABEL_NAMES[i] for i in preds],
        }).to_csv(out_dir / f"predicciones_{name}.csv", index=False)
    return results


def save_summary(out_dir, **summary):
    (out_dir / "resumen.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))


def load_summary(out_dir):
    path = out_dir / "resumen.json"
    return json.loads(path.read_text()) if SKIP_DONE and path.exists() else None


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 7. BiLSTM bilingüe

Misma arquitectura e hiperparámetros que el BiLSTM final de la etapa 1. El vocabulario se construye con el train bilingüe, así que incluye palabras en español; GloVe Twitter trae vectores para parte de ellas (Twitter es multilingüe) y el resto se aprende durante el entrenamiento. El modelo queda en `models/etapa2/bilstm_bi/`.

In [ ]:
PAD, UNK = "<pad>", "<unk>"
TOKEN_RE = re.compile(r"<user>|<url>|[a-z0-9áéíóúüñ]+(?:'[a-z]+)?|[^\sa-z0-9áéíóúüñ]")
NUMBER_RE = re.compile(r"^\d+$")
GLOVE_REPO = "stanfordnlp/glove"
GLOVE_ZIP = "glove.twitter.27B.zip"
GLOVE_DIM = 200
MIN_FREQUENCY = 2


def tokenize(text):
    text = text.replace("@USER", " <user> ").replace("HTTPURL", " <url> ").lower()
    return ["<number>" if NUMBER_RE.match(tok) else tok for tok in TOKEN_RE.findall(text)]


def build_vocab(texts, min_frequency):
    counts = Counter(tok for text in texts for tok in tokenize(text))
    words = sorted((w for w, c in counts.items() if c >= min_frequency), key=lambda w: (-counts[w], w))
    return {word: i for i, word in enumerate([PAD, UNK] + words)}


def download_glove():
    return hf_hub_download(GLOVE_REPO, GLOVE_ZIP, local_dir=GLOVE_DIR)


# Embedding matrix: GloVe vector when the word exists, random with the same scale otherwise.
def glove_matrix(vocab, dim):
    found = {}
    with zipfile.ZipFile(download_glove()) as zf, zf.open(f"glove.twitter.27B.{dim}d.txt") as f:
        for line in io.TextIOWrapper(f, encoding="utf-8"):
            parts = line.rstrip().split(" ")
            if len(parts) == dim + 1 and parts[0] in vocab:
                found[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    std = np.std(np.stack(list(found.values()))) if found else 0.1
    rng = np.random.default_rng(SEED)
    matrix = rng.normal(0, std, size=(len(vocab), dim)).astype(np.float32)
    for word, vec in found.items():
        matrix[vocab[word]] = vec
    matrix[vocab[PAD]] = 0.0
    return matrix, len(found) / (len(vocab) - 2)


VOCAB = build_vocab(train_df[TEXT_COL], MIN_FREQUENCY)
PAD_ID, UNK_ID = VOCAB[PAD], VOCAB[UNK]
GLOVE_MATRIX, glove_coverage = glove_matrix(VOCAB, GLOVE_DIM)
print(f"Vocabulario: {len(VOCAB):,} | con vector GloVe: {glove_coverage:.1%}")
print(tokenize(train_df[TEXT_COL].iloc[0]))

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.ids = [[VOCAB.get(tok, UNK_ID) for tok in tokenize(text)][:max_len] or [UNK_ID] for text in texts]
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.ids[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


def scratch_loader_factory(hp):
    def collate_fn(batch):
        sequences, labels = zip(*batch)
        lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
        padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
        return padded, torch.stack(labels), lengths

    def make_loader(df, shuffle=False):
        dataset = TokenDataset(df[TEXT_COL], df["label_id"], hp["max_len"])
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle, collate_fn=collate_fn)
    return make_loader

In [ ]:
def make_embedding(hp):
    embedding = nn.Embedding(len(VOCAB), GLOVE_DIM, padding_idx=PAD_ID)
    if hp["embeddings"] == "glove":
        embedding.weight.data.copy_(torch.from_numpy(GLOVE_MATRIX))
        embedding.weight.requires_grad = not hp["freeze_embeddings"]
    return embedding


def mlp_head(in_dim, hidden, num_classes, dropout):
    return nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, hidden), nn.ReLU(),
                         nn.Dropout(dropout), nn.Linear(hidden, num_classes))


# True for real tokens, False for padding.
def token_mask(ids, lengths):
    return torch.arange(ids.size(1), device=ids.device)[None, :] < lengths[:, None]


class BiLSTMClassifier(nn.Module):
    def __init__(self, hp, num_classes):
        super().__init__()
        self.pooling = hp["pooling"]
        self.embedding = make_embedding(hp)
        self.emb_dropout = nn.Dropout(hp["emb_dropout"])
        self.lstm = nn.LSTM(GLOVE_DIM, hp["hidden_size"], num_layers=hp["num_layers"], batch_first=True,
                            bidirectional=True, dropout=hp["rnn_dropout"] if hp["num_layers"] > 1 else 0.0)
        # Being bidirectional doubles the size of the LSTM output.
        out_dim = 2 * hp["hidden_size"]
        if "attention" in self.pooling:
            self.attention = nn.Linear(out_dim, 1)
        rep_dim = out_dim * (2 if self.pooling == "attention_max" else 1)
        self.head = mlp_head(rep_dim, hp["head_hidden"], num_classes, hp["dropout"])

    def forward(self, ids, lengths):
        emb = self.emb_dropout(self.embedding(ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=ids.size(1))
        mask = token_mask(ids, lengths)
        parts = []
        if "attention" in self.pooling:
            scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
            parts.append((torch.softmax(scores, dim=1).unsqueeze(-1) * out).sum(dim=1))
        if "max" in self.pooling:
            parts.append(out.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(dim=1).values)
        return self.head(torch.cat(parts, dim=1))


def forward_scratch(model, batch):
    ids, labels, lengths = batch
    return model(ids.to(DEVICE), lengths.to(DEVICE)), labels.to(DEVICE)

In [ ]:
# Trains one configuration; returns the best model (by val F1-macro) and its training info.
def train_scratch(hp, run_name, out_dir, seed):
    set_all_seeds(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = MODELS_DIR / f"{run_name}.pt"
    make_loader = scratch_loader_factory(hp)
    train_loader, val_loader = make_loader(train_df, shuffle=True), make_loader(val_df)

    model = BiLSTMClassifier(hp, NUM_CLASSES).to(DEVICE)
    print(f"\n[{run_name}] parámetros entrenables: {count_parameters(model)['entrenables']:,}")
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=hp["lr"], weight_decay=hp["weight_decay"])
    checkpoint = ModelCheckpoint(str(ckpt_path))
    reduce_lr = ReduceLROnPlateau(optimizer, patience=hp["lr_patience"], factor=hp["lr_factor"])
    early_stop = EarlyStopping(patience=hp["patience"])

    history = new_history()
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, hp["epochs"] + 1):
        train_m = run_epoch(model, train_loader, criterion, forward_scratch, optimizer, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_scratch)
        metrics = log_epoch(history, epoch, "bilstm", optimizer.param_groups[0]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        reduce_lr.step(metrics)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, run_name, out_dir)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return {"model": model, "make_loader": make_loader, "recursos": resources,
            "epocas": len(history["epoch"]), "val_f1_macro": checkpoint.best}


# Final run: trains with SEED, evaluates on every set and saves the model in EXPORT_DIR.
def final_scratch(hp, family):
    out_dir, model_dir = RESULTS_DIR / family, EXPORT_DIR / family
    if load_summary(out_dir) and (model_dir / "model.pt").exists():
        print(f"{family}: ya tiene resultados y modelo guardado, se omite")
        return
    run = train_scratch(hp, family, out_dir, SEED)
    results = final_evaluation(run["model"], run["make_loader"], forward_scratch, family, out_dir)
    save_summary(out_dir, nombre=family, familia=family, semilla=SEED, tipo="desde_cero",
                 mejor_val_f1_macro=run["val_f1_macro"], epocas_entrenadas=run["epocas"],
                 parametros=count_parameters(run["model"]), vocabulario=len(VOCAB),
                 cobertura_glove=glove_coverage, recursos=run["recursos"], resultados=results, hiperparametros=hp)
    save_bilstm(run["model"], hp, model_dir, run["val_f1_macro"])
    (MODELS_DIR / f"{family}.pt").unlink(missing_ok=True)
    del run
    free_memory()


# Everything needed to rebuild the BiLSTM outside the notebook: weights, vocabulary and configuration.
def save_bilstm(model, hp, model_dir, val_f1):
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_dir / "model.pt")
    (model_dir / "vocab.json").write_text(json.dumps(VOCAB, ensure_ascii=False))
    config = {"tipo": "bilstm", "semilla": SEED, "mejor_val_f1_macro": val_f1, "labels": LABEL_NAMES,
              "glove_dim": GLOVE_DIM, "hiperparametros": hp}
    (model_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
    print(f"Modelo guardado en {model_dir}")

In [ ]:
if RUN["bilstm"]:
    final_scratch(BILSTM_HP, "bilstm_bi")

## 8. Transformer bilingüe

El ganador del screening de la etapa 1, con la misma receta (mean pooling, LP-FT, LLRD, early stopping) y los mismos hiperparámetros. El modelo queda en `models/etapa2/<modelo>/`.

In [ ]:
class PretrainedTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in ("input_ids", "attention_mask")}
        item["labels"] = self.labels[idx]
        return item


class PretrainedClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        # Always fp32 weights (some checkpoints load as fp16 by default); autocast handles mixed precision.
        self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_classes)

    # Mean of the token vectors, ignoring padding.
    def pool(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    def forward(self, input_ids, attention_mask):
        return self.classifier(self.drop(self.pool(input_ids, attention_mask)))

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


def forward_transformer(model, batch):
    logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
    return logits, batch["labels"].to(DEVICE)

In [ ]:
# Layer-wise lr decay: the top layer gets lr, each layer below gets lr * decay; no weight decay on biases or LayerNorm.
def llrd_param_groups(model, lr, decay, weight_decay):
    num_layers = model.encoder.config.num_hidden_layers
    groups = {}
    for name, param in model.named_parameters():
        if name.startswith("encoder.embeddings."):
            depth = 0
        elif match := re.match(r"encoder\.encoder\.layer\.(\d+)\.", name):
            depth = int(match.group(1)) + 1
        else:
            depth = num_layers + 1
        no_decay = param.ndim == 1
        group = groups.setdefault((depth, no_decay), {
            "params": [], "lr": lr * decay ** (num_layers + 1 - depth),
            "weight_decay": 0.0 if no_decay else weight_decay,
        })
        group["params"].append(param)
    return list(groups.values())


# Pooled vectors of the frozen encoder, computed once for the linear probe.
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, labels = [], []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            pooled = model.pool(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        feats.append(pooled.float())
        labels.append(batch["labels"].to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


# Phase 1 (linear probe): trains only the classifier on the cached vectors until val F1 stops improving.
def train_head(model, train_feats, val_feats, criterion, hp, history):
    head_params = list(model.classifier.parameters())
    optimizer = torch.optim.Adam(head_params, lr=hp["lp_lr"])
    early_stop = EarlyStopping(patience=hp["lp_patience"])
    best_f1, best_state = float("-inf"), None
    (x_train, y_train), (x_val, y_val) = train_feats, val_feats
    for epoch in range(1, hp["lp_epochs"] + 1):
        model.drop.train()
        order = torch.randperm(len(y_train), device=DEVICE)
        total, preds = 0.0, torch.empty_like(y_train)
        for start in range(0, len(order), 256):
            idx = order[start:start + 256]
            logits = model.classifier(model.drop(x_train[idx]))
            loss = criterion(logits, y_train[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
            preds[idx] = logits.argmax(dim=1)
        model.drop.eval()
        with torch.no_grad():
            val_logits = model.classifier(x_val)
            val_loss = criterion(val_logits, y_val).item()
        train_m = compute_metrics(y_train.tolist(), preds.tolist())
        val_m = compute_metrics(y_val.tolist(), val_logits.argmax(dim=1).tolist())
        metrics = log_epoch(history, epoch, "lp", hp["lp_lr"],
                            (total / len(y_train), train_m["accuracy"], train_m["f1_macro"]),
                            (val_loss, val_m["accuracy"], val_m["f1_macro"]))
        if metrics["val_f1_macro"] > best_f1:
            best_f1 = metrics["val_f1_macro"]
            best_state = {k: v.detach().clone() for k, v in model.classifier.state_dict().items()}
        early_stop.step(metrics)
        if early_stop.should_stop:
            break
    model.classifier.load_state_dict(best_state)
    return best_f1

In [ ]:
# Everything needed to rebuild the classifier outside the notebook: weights, encoder configuration and tokenizer.
def save_transformer(model, tokenizer, hp, model_dir, val_f1, seed):
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_dir / "model.pt")
    model.encoder.config.save_pretrained(model_dir / "encoder")
    tokenizer.save_pretrained(model_dir / "tokenizer")
    config = {"tipo": "transformer", "semilla": seed, "mejor_val_f1_macro": val_f1, "labels": LABEL_NAMES,
              "hiperparametros": hp}
    (model_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
    print(f"Modelo guardado en {model_dir}")


# LP-FT: linear probe, then the whole model with LLRD; the best model by val F1-macro is kept.
def finetune(hp, run_name, out_dir, seed, full_eval, export_dir=None):
    set_all_seeds(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = MODELS_DIR / f"{run_name}.pt"
    use_amp = hp["amp"] and DEVICE.type == "cuda"

    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"], **hp.get("tokenizer_kwargs", {}))
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    num_workers = 2 if DEVICE.type == "cuda" else 0

    def make_loader(df, shuffle=False):
        dataset = PretrainedTweetDataset(df[TEXT_COL], df["label_id"], tokenizer, hp["max_len"])
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle,
                          collate_fn=collator, num_workers=num_workers)

    train_loader, val_loader = make_loader(train_df, shuffle=True), make_loader(val_df)
    model = PretrainedClassifier(hp["model_name"], NUM_CLASSES, dropout=hp["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    checkpoint = ModelCheckpoint(str(ckpt_path))
    history = new_history()
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    print(f"\n[{run_name}] Fase 1 — encoder congelado (linear probe)")
    model.freeze_encoder()
    train_feats = extract_features(model, make_loader(train_df))
    val_feats = extract_features(model, val_loader)
    lp_f1 = train_head(model, train_feats, val_feats, criterion, hp, history)
    del train_feats, val_feats
    checkpoint.step({"val_f1_macro": lp_f1}, model)
    lp_epochs = len(history["epoch"])

    model.unfreeze_encoder()
    print(f"\n[{run_name}] Fase 2 — fine-tuning completo, {count_parameters(model)['entrenables']:,} parámetros")
    optimizer = torch.optim.AdamW(llrd_param_groups(model, hp["lr"], hp["llrd"], hp["weight_decay"]))
    total_steps = len(train_loader) * hp["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * hp["warmup_ratio"]), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    early_stop = EarlyStopping(patience=hp["patience"])
    for epoch in range(1, hp["epochs"] + 1):
        train_m = run_epoch(model, train_loader, criterion, forward_transformer, optimizer,
                            scheduler=scheduler, scaler=scaler, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
        metrics = log_epoch(history, lp_epochs + epoch, "ft", optimizer.param_groups[-1]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, run_name, out_dir)
    summary = {"nombre": run_name, "modelo": hp["model_name"], "mejor_val_f1_macro": checkpoint.best,
               "val_f1_macro_lp": lp_f1, "epocas_lp": lp_epochs, "epocas_ft": len(history["epoch"]) - lp_epochs,
               "epocas_entrenadas": len(history["epoch"]), "parametros": count_parameters(model),
               "recursos": resources, "hiperparametros": hp}
    if full_eval or export_dir is not None:
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    if full_eval:
        summary["resultados"] = final_evaluation(model, make_loader, forward_transformer, run_name, out_dir)
    save_summary(out_dir, **summary)
    if export_dir is not None:
        save_transformer(model, tokenizer, hp, export_dir, checkpoint.best, seed)
    print(f"Recursos: {resources}")

    del model, optimizer
    ckpt_path.unlink(missing_ok=True)
    free_memory()
    return summary

In [ ]:
# Final run: trains with SEED and evaluates on every set; with export, the model is saved in EXPORT_DIR.
def final_finetune(hp, family, export=True):
    out_dir, model_dir = RESULTS_DIR / family, EXPORT_DIR / family
    if load_summary(out_dir) and (not export or (model_dir / "model.pt").exists()):
        print(f"{family}: ya tiene resultados, se omite")
        return
    summary = finetune(hp, family, out_dir, SEED, full_eval=True, export_dir=model_dir if export else None)
    save_summary(out_dir, **summary, familia=family, semilla=SEED, tipo="fine_tuning")


if RUN["transformer"]:
    final_finetune(FT_HP, f"{FT_KEY}_bi")

## 9. Comparación con la etapa 1

`comparacion.csv` tiene una fila por modelo final. `comparacion_etapas.csv` junta los modelos finales de ambas etapas, y la última tabla muestra el cambio de la etapa 1 a la 2 (positivo = mejor en la etapa 2). En la etapa 2, `val`, `test` y `test_intra` incluyen también su traducción, así que ese cambio mezcla el efecto del modelo con el de evaluar en dos idiomas; `chile_2014` es idéntico en ambas etapas y es la comparación directa.

In [ ]:
def runs_table(results_dir, stage):
    rows = []
    for path in sorted(results_dir.glob("*/resumen.json")):
        summary = json.loads(path.read_text())
        if "resultados" not in summary:
            continue
        row = {"etapa": stage, "modelo": summary["familia"], "tipo": summary["tipo"],
               "épocas": summary["epocas_entrenadas"], "minutos": summary["recursos"]["tiempo_entrenamiento_min"],
               "gpu": summary["recursos"]["gpu"]}
        row.update({f"F1 {name}": round(r["f1_macro"], 4) for name, r in summary["resultados"].items()})
        row.update({f"F1 ponderado {name}": round(r["f1_weighted"], 4) for name, r in summary["resultados"].items()})
        rows.append(row)
    return pd.DataFrame(rows)


stage2 = runs_table(RESULTS_DIR, 2)
stage2.drop(columns="etapa").to_csv(RESULTS_DIR / "comparacion.csv", index=False)

stage1 = runs_table(STAGE1_DIR, 1)
both = pd.concat([stage1, stage2], ignore_index=True)
both.to_csv(RESULTS_DIR / "comparacion_etapas.csv", index=False)
both

In [ ]:
# Change from stage 1 to stage 2 for each model family.
common = ["F1 val", "F1 test", "F1 test_intra", "F1 chile_2014"]
means = both.set_index(["etapa", "modelo"])[common]
pairs = [("bilstm_en", "bilstm_bi"), (f"{FT_KEY}_en", f"{FT_KEY}_bi")]
deltas = pd.DataFrame([
    {"etapa 1": a, "etapa 2": b, **{m: round(means.loc[(2, b), m] - means.loc[(1, a), m], 4) for m in common}}
    for a, b in pairs if (1, a) in means.index and (2, b) in means.index
])
deltas

In [ ]:
if Path("/kaggle").exists():
    shutil.make_archive("/kaggle/working/resultados_etapa2", "zip", RESULTS_DIR)
    shutil.make_archive("/kaggle/working/modelos_etapa2", "zip", EXPORT_DIR)
    print("Resultados en /kaggle/working/resultados_etapa2.zip y modelos en /kaggle/working/modelos_etapa2.zip")